# One Recurrent PPO update you can reproduce
**Quadwind · Worked example for the project defense**

Follow three consecutive decisions of a one-axis hover agent. You will see the sampled actions, the LSTM memory, the rewards and the exact change to both the policy and the value estimate after **one optimizer step**. The example runs on CPU in a few seconds with PyTorch. Open it in a Python 3 Jupyter kernel and choose **Restart Kernel and Run All Cells**. Saved outputs are included.

The drone simulator uses four continuous action channels and separate actor and critic LSTMs. Here each network has one memory cell and the action is one continuous correction. A small, finite-horizon simulator makes it possible to check the complete calculation by hand.

## 1 · Define the policy and the one-axis task
The measured position error is $x$; zero is the target. A constant hidden wind $w=0.2$ adds drift. The simulator advances $x' = x + 0.2(a+w)$ and returns $r=1-|x'|-0.05a^2$. The episode truly ends after decision 3. Use $x_0=0.4$.

The actor outputs the mean $\mu$ of a Gaussian with fixed standard deviation $\sigma=0.3$. The critic outputs $V$, a prediction of discounted future reward. Each uses a separate scalar LSTM. We fix the input, forget and output gates at $\operatorname{sigmoid}(0)=0.5$ so that the cell can be unrolled on paper. Its candidate and output weights remain trainable.
$$g_t=\tanh(w_x x_t+w_h h_{t-1}),\quad c_t=0.5 c_{t-1}+0.5 g_t,\quad h_t=0.5\tanh(c_t),\quad y_t=k h_t+b.$$
The initial states are zero. The actor uses $y_t=\mu_t$; the critic uses $y_t=V_t$.

In [1]:
import torch
from torch import nn
from torch.distributions import Normal

dtype = torch.float64
def scalar(value):
    return torch.tensor(float(value), dtype=dtype)

class ScalarLSTM(nn.Module):
    def __init__(self, input_weight, recurrent_weight, output_weight, bias):
        super().__init__()
        self.wx = nn.Parameter(scalar(input_weight))
        self.wh = nn.Parameter(scalar(recurrent_weight))
        self.k = nn.Parameter(scalar(output_weight))
        self.bias = nn.Parameter(scalar(bias))

    def forward(self, x, state):
        h_prev, c_prev = state
        candidate = torch.tanh(self.wx * x + self.wh * h_prev)
        cell = 0.5 * c_prev + 0.5 * candidate
        hidden = 0.5 * torch.tanh(cell)
        return self.k * hidden + self.bias, (hidden, cell)

def start_state():
    return scalar(0), scalar(0)

actor = ScalarLSTM(1.0, 0.5, -0.8, 0.05)
critic = ScalarLSTM(0.5, 0.3, -0.6, 0.2)
sigma = scalar(0.3)
gamma, gae_lambda, epsilon = 0.99, 0.95, 0.2
learning_rate = 0.10

def transition(x, action):
    x_next = x + 0.2 * (action + 0.2)
    reward = 1 - x_next.abs() - 0.05 * action.square()
    return x_next, reward

## 2 · Collect and freeze one rollout
For each decision, sample $a_t=\mu_t+\sigma z_t$ with $z_t\sim\mathcal N(0,1)$, then store the observation, sampled action, reward, old value and **old log-density**. For continuous actions, a probability at one exact point is zero; PPO compares densities. The fixed seed reproduces the same three draws. Both memories carry across the three decisions. Rollout collection uses frozen weights.

In [2]:
rng = torch.Generator().manual_seed(7)
observations, actions, rewards, old_values, old_logp = [], [], [], [], []
x = scalar(0.4)
actor_state, critic_state = start_state(), start_state()
print(" t       x       c       h      mu       a      r  V_old   logp_old")
with torch.no_grad():
    for t in range(3):
        mean, actor_state = actor(x, actor_state)
        value, critic_state = critic(x, critic_state)
        noise = torch.randn((), generator=rng, dtype=dtype)
        action = mean + sigma * noise
        logp = Normal(mean, sigma).log_prob(action)
        x_next, reward = transition(x, action)
        observations.append(x)
        actions.append(action)
        rewards.append(reward)
        old_values.append(value)
        old_logp.append(logp)
        h, cell = actor_state
        print(f"{t:2d} {x:7.4f} {cell:7.4f} {h:7.4f} {mean:7.4f} "
              f"{action:7.4f} {reward:6.4f} {value:6.4f} {logp:10.4f}")
        x = x_next
observations, actions, rewards, old_values, old_logp = map(
    torch.stack, (observations, actions, rewards, old_values, old_logp)
)
assert not old_logp.requires_grad and not old_values.requires_grad

 t       x       c       h      mu       a      r  V_old   logp_old
 0  0.4000  0.1900  0.0939 -0.0251 -0.0691 0.5736 0.1705     0.2743
 1  0.4262  0.3153  0.1526 -0.0721  0.1637 0.4997 0.1520    -0.0240
 2  0.4989  0.4173  0.1973 -0.1079  0.1762 0.4243 0.1367    -0.1632


## 3 · Assign credit with generalized advantage estimation (GAE)
The temporal-difference residual is $\delta_t=r_t+\gamma V_{t+1}-V_t$. Starting at the end, accumulate $A_t=\delta_t+\gamma\lambda A_{t+1}$. At a true terminal boundary both the next value and the next advantage are zero. At a rollout cutoff in an ongoing episode, bootstrap from the critic's next value. Keep targets fixed while optimizing this rollout.

Use $R_t=A_t+V_t$ as the critic target. For the policy loss only, standardize the three advantages with the sample standard deviation, as in the project trainer. A negative standardized advantage identifies a decision with lower advantage than the batch average.

In [3]:
next_values = torch.cat((old_values[1:], scalar(0).reshape(1)))
delta = rewards + gamma * next_values - old_values
advantage = torch.empty_like(delta)
carry = scalar(0)
for t in reversed(range(len(delta))):
    carry = delta[t] + gamma * gae_lambda * carry
    advantage[t] = carry
returns = advantage + old_values
adv_policy = (advantage - advantage.mean()) / (advantage.std() + 1e-8)
assert torch.isclose(returns[-1], rewards[-1])
print(" t    delta    A_raw   target   A_norm")
for t in range(len(rewards)):
    print(f"{t:2d} {delta[t]:8.4f} {advantage[t]:8.4f} "
          f"{returns[t]:8.4f} {adv_policy[t]:8.4f}")

 t    delta    A_raw   target   A_norm
 0   0.5536   1.2623   1.4328   1.0143
 1   0.4831   0.7535   0.9056  -0.0292
 2   0.2876   0.2876   0.4243  -0.9851


## 4 · Replay the same sequence and evaluate the clipped objective
Start from the same initial memories and replay the **stored observations and actions** in chronological order. During replay, gradients can flow through the LSTM across all three time steps.

$$q_t=\exp(\log\pi_{\theta}(a_t\mid h_t)-\log\pi_{\rm old}(a_t\mid h_t^{\rm old})),\qquad
s_t=\min\{q_t\hat A_t,\operatorname{clip}(q_t,0.8,1.2)\hat A_t\}.$$
The policy loss is $L_\pi=-\mathrm{mean}(s_t)$. The critic loss is $L_V=\mathrm{mean}((V_t-R_t)^2)$; the combined objective minimized here is $L=L_\pi+0.5L_V$. The entropy bonus is zero. At the beginning every ratio is one because the weights and initial memory match the collecting policy. A zero-valued policy loss can still have a nonzero derivative.

In [4]:
def replay(network):
    state = start_state()
    predictions = []
    for observation in observations:
        prediction, state = network(observation, state)
        predictions.append(prediction)
    return torch.stack(predictions)

def objective():
    means = replay(actor)
    values = replay(critic)
    logp = Normal(means, sigma).log_prob(actions)
    ratio = (logp - old_logp).exp()
    direct = ratio * adv_policy
    clipped = ratio.clamp(1 - epsilon, 1 + epsilon) * adv_policy
    surrogate = torch.minimum(direct, clipped)
    policy_loss = -surrogate.mean()
    value_loss = (values - returns).square().mean()
    return policy_loss + 0.5 * value_loss, policy_loss, value_loss, ratio, surrogate

total, policy_loss, value_loss, ratio, surrogate = objective()
assert torch.allclose(ratio, torch.ones_like(ratio))
print("ratios:", ratio.detach().tolist())
print(f"actor loss {policy_loss.item():.6f}, critic MSE {value_loss.item():.6f}, "
      f"total {total.item():.6f}")

ratios: [1.0, 1.0, 1.0]
actor loss 0.000000, critic MSE 0.747962, total 0.373981


## 5 · Take exactly one optimizer step
Differentiate the total loss through both unrolled sequences. We use plain SGD and a fixed learning rate here so each change can be checked with $\theta'=\theta-\eta\,\partial L/\partial\theta$. For the Gaussian actor's output bias, $\partial\log\pi/\partial b=(a-\mu)/\sigma^2$. At the starting policy the ratio is one, so an independent check is $\partial L/\partial b=-\mathrm{mean}(\hat A(a-\mu)/\sigma^2)$.

The final row reports a **recurrent** weight to show that the gradient also reaches memory.

In [5]:
parameters = list(actor.parameters()) + list(critic.parameters())
optimizer = torch.optim.SGD(parameters, lr=learning_rate)
optimizer.zero_grad()
mean_before = replay(actor).detach()
value_before = replay(critic).detach()
total.backward()
manual_bias_grad = -(adv_policy * (actions - mean_before) / sigma.square()).mean()
assert torch.isclose(actor.bias.grad, manual_bias_grad)
tracked = {"actor.bias": actor.bias, "critic.bias": critic.bias, "actor.wh": actor.wh}
before = {label: p.detach().clone() for label, p in tracked.items()}
optimizer.step()
print("parameter       before    dL/dp     after")
for label, p in tracked.items():
    expected = before[label] - learning_rate * p.grad
    assert torch.isclose(p.detach(), expected)
    print(f"{label:<13s} {before[label].item():8.5f} {p.grad.item():8.5f} {p.item():8.5f}")

parameter       before    dL/dp     after
actor.bias     0.05000  1.22729 -0.07273
critic.bias    0.20000 -0.76779  0.27678
actor.wh       0.50000 -0.02757  0.50276


## 6 · Measure the effect on the original three decisions
Recompute predictions on the same observations after the step. Inspect the density ratios of the *same recorded actions*. The last two columns compare the unconstrained contribution with the one PPO would use if another epoch started now. This cell performs evaluation only. PPO clipping flattens a term once a change is already favorable and exceeds its threshold. It does not impose a hard bound on the resulting ratio.

In [6]:
with torch.no_grad():
    new_total, new_pi, new_vf, new_ratio, new_surr = objective()
    mean_after, value_after = replay(actor), replay(critic)
    print(" t  mu_old  mu_new   V_old   V_new   ratio  ratio*A  PPO term")
    for t in range(len(actions)):
        print(f"{t:2d} {mean_before[t]:7.4f} {mean_after[t]:7.4f} "
              f"{value_before[t]:7.4f} {value_after[t]:7.4f} {new_ratio[t]:7.4f} "
              f"{(new_ratio[t] * adv_policy[t]):8.4f} {new_surr[t]:9.4f}")
    print(f"total loss {total.item():.6f} -> {new_total.item():.6f}")
    print(f"critic MSE {value_loss.item():.6f} -> {new_vf.item():.6f}")
    assert not torch.allclose(mean_before, mean_after)
    assert not torch.allclose(value_before, value_after)
    assert new_total < total and new_vf < value_loss

 t  mu_old  mu_new   V_old   V_new   ratio  ratio*A  PPO term
 0 -0.0251 -0.1508  0.1705  0.2478  0.9740   0.9880    0.9880
 1 -0.0721 -0.1997  0.1520  0.2297  0.6540  -0.0191   -0.0234
 2 -0.1079 -0.2366  0.1367  0.2147  0.6074  -0.5983   -0.7880
total loss 0.373981 -> 0.258609
critic MSE 0.747962 -> 0.634905


### Read the update aloud

At the first observation, the actor's mean command moved from **−0.0251 to −0.1508**. The critic raised its value estimate **from 0.1705 to 0.2478** toward the fixed target **1.4328**. Across this particular trajectory, the critic MSE fell from **0.747962 to 0.634905**. Performance on fresh episodes is a separate measurement.

At the final observation, the same recorded action now has a density ratio of **0.6074** and a normalized advantage of **−0.9851**. The raw surrogate contribution is **−0.5983**. The lower clipping boundary gives **0.8 × (−0.9851) ≈ −0.7880**. PPO chooses the smaller of these contributions. This caps the gain a further epoch could obtain from reducing that action’s density.

**Try it.** Restart the kernel and reduce the learning rate in section 1 from 0.10 to 0.01. Predict which final ratios will move closer to one. Compare the resulting table with the saved run, then restore 0.10.

## Connecting the calculation to Quadwind

| Component | This notebook | Trained quadrotor agent |
|---|---|---|
| Rollout | 1 episode, 3 decisions, scalar position | 16 environments × 512 decisions per rollout, 50 Hz control |
| Policy | One scalar LSTM and a Gaussian correction | 128-unit actor LSTM; four Gaussian action components for thrust, roll, pitch and yaw rate |
| Critic | Separate scalar LSTM | Separate 128-unit critic LSTM with extra simulator features including true wind and mass |
| Update | One chronological batch, one SGD step, fixed variance | Adam, five epochs, batch size 4096, learned log standard deviations and gradient norm bound 0.5 |
| Loss | $\gamma=0.99,\lambda=0.95,\epsilon=0.2$; normalized advantages | Same factors and advantage normalization; observation and reward normalization during training |

**At inference**, the actor carries $(h,c)$ from one decision to the next. A new episode resets them. During training the recurrent buffer saves sequence-start states, keeps the time order inside each sequence and masks padding. The optimizer differentiates through those sequences. In the four-channel policy the joint log-density is the sum of the four component log-densities.

**Implementation.** `quadwind/training.py` builds the trainer using SB3-Contrib RecurrentPPO. `quadwind/asymmetric.py` selects the inputs available to the actor and critic. `quadwind/fast_lstm.py` handles recurrent sequences. The small networks and update equations in this notebook are implemented directly with PyTorch.

**Reading.** [PPO, Schulman et al.](https://arxiv.org/abs/1707.06347) · [Generalized advantage estimation, Schulman et al.](https://arxiv.org/abs/1506.02438) · [SB3-Contrib recurrent policies](https://sb3-contrib.readthedocs.io/en/master/modules/ppo_recurrent.html)